# Aspectos preliminares

Este script tiene por objetivo verificar si existe solapamiento de tablas de dimensiones a partir de la normalización de las tablas de transacciones.

In [1]:
# Importacion de librerias.
import pandas as pd
import numpy as np
import os, zipfile

# Ejecución del control

In [8]:
# Inicializacion de variables.
lookup_dir = '../output/'
excluded_folder = 'application_test'

In [9]:
# Verificación de coincidencia de archivos entre carpetas.

def find_duplicate_filenames(lookup_dir='.', exclude_folder='sample_data'):
    """
    Busca archivos con el mismo nombre en diferentes carpetas del directorio actual,
    excluyendo una carpeta específica.

    Args:
        lookup_dir (str): Ruta del directorio donde buscar las carpetas.
        exclude_folder (str): Nombre de la carpeta a excluir de la búsqueda.
    """
    current_directory = lookup_dir
    all_folders = [f for f in os.listdir(current_directory) if os.path.isdir(os.path.join(current_directory, f)) and f != exclude_folder]

    if not all_folders:
        print(f"No se encontraron carpetas para analizar (excluyendo {exclude_folder}).")
        return

    print(f"Carpetas a analizar: {', '.join(all_folders)}\n")

    # Diccionario para almacenar los archivos encontrados y las carpetas donde aparecen
    files_in_folders = {}

    for folder in all_folders:
        folder_path = os.path.join(current_directory, folder)
        try:
            for item in os.listdir(folder_path):
                # Solo consideramos archivos, no subcarpetas
                if os.path.isfile(os.path.join(folder_path, item)):
                    if item not in files_in_folders:
                        files_in_folders[item] = []
                    files_in_folders[item].append(folder)
        except Exception as e:
            print(f"Error al leer la carpeta '{folder}': {e}")

    duplicates_found = False
    for filename, folders_list in files_in_folders.items():
        if len(folders_list) > 1:
            print(f"Archivo duplicado: '{filename}'")
            print(f"Se encuentra en las carpetas: {', '.join(folders_list)}\n")
            duplicates_found = True

    if not duplicates_found:
        print("No se encontraron archivos duplicados entre las carpetas analizadas.")

# Ejecutar la función
find_duplicate_filenames(lookup_dir=lookup_dir, exclude_folder=excluded_folder)


Carpetas a analizar: application_train, bureau, previous_application

Archivo duplicado: 'dim_name_contract_type.csv'
Se encuentra en las carpetas: application_train, previous_application

Archivo duplicado: 'dim_name_type_suite.csv'
Se encuentra en las carpetas: application_train, previous_application

Archivo duplicado: 'dim_weekday_appr_process_start.csv'
Se encuentra en las carpetas: application_train, previous_application



Archivo duplicado: 'dim_name_contract_type.csv'


In [12]:
# Head comparativo de tablas.
csv_name = 'dim_name_contract_type.csv'

display(pd.read_csv(lookup_dir + 'application_train' + '/' + csv_name))
display(pd.read_csv(lookup_dir + 'previous_application' + '/' + csv_name))

,NAME_CONTRACT_TYPE_ID,NAME_CONTRACT_TYPE
0,1,Cash loans
1,2,Revolving loans


,NAME_CONTRACT_TYPE_ID,NAME_CONTRACT_TYPE
0,1,Consumer loans
1,2,Cash loans
2,3,Revolving loans
3,4,XNA


Archivo duplicado: 'dim_weekday_appr_process_start.csv'


In [13]:
# Head comparativo de tablas.
csv_name = 'dim_weekday_appr_process_start.csv'

display(pd.read_csv(lookup_dir + 'application_train' + '/' + csv_name))
display(pd.read_csv(lookup_dir + 'previous_application' + '/' + csv_name))

,WEEKDAY_APPR_PROCESS_START_ID,WEEKDAY_APPR_PROCESS_START
0,1,WEDNESDAY
1,2,MONDAY
2,3,THURSDAY
3,4,SUNDAY
4,5,SATURDAY
5,6,FRIDAY
6,7,TUESDAY


,WEEKDAY_APPR_PROCESS_START_ID,WEEKDAY_APPR_PROCESS_START
0,1,SATURDAY
1,2,THURSDAY
2,3,TUESDAY
3,4,MONDAY
4,5,FRIDAY
5,6,SUNDAY
6,7,WEDNESDAY


Archivo duplicado: 'dim_name_type_suite.csv'


In [14]:
# Head comparativo de tablas.
csv_name = 'dim_name_type_suite.csv'

display(pd.read_csv(lookup_dir + 'application_train' + '/' + csv_name))
display(pd.read_csv(lookup_dir + 'previous_application' + '/' + csv_name))

,NAME_TYPE_SUITE_ID,NAME_TYPE_SUITE
0,1,Unaccompanied
1,2,Family
2,3,"Spouse, partner"
3,4,Children
4,5,Other_A
5,6,Other_B
6,7,Group of people


,NAME_TYPE_SUITE_ID,NAME_TYPE_SUITE
0,1,Unaccompanied
1,2,"Spouse, partner"
2,3,Family
3,4,Children
4,5,Other_B
5,6,Other_A
6,7,Group of people


De la ejecución del test, se observa que existen algunas categorías y órdenes diferentes en la asignación de ID para cada tabla de transacciones.

# Ejecución de correcciones

### Tablas de dimensiones de referencia

Se establecen las tres tablas de dimensiones de referencia.

Para la tabla dim_name_contract_type, se toma como referencia las categorías de la tabla previous_application.

In [16]:
# Categorias de la tabla dim_name_contract_type.
csv_name = 'dim_name_contract_type.csv'
ref_table = 'previous_application'
dim_name_contract_type_ref = pd.read_csv(lookup_dir + ref_table + '/' + csv_name)
display(dim_name_contract_type_ref)

,NAME_CONTRACT_TYPE_ID,NAME_CONTRACT_TYPE
0,1,Consumer loans
1,2,Cash loans
2,3,Revolving loans
3,4,XNA


In [17]:
# Se creará una tabla `dim_weekday_appr_process_start` estandarizada con `WEEKDAY_APPR_PROCESS_START_ID` del 1 (Lunes) al 7 (Domingo).

# Creación de la tabla dim_weekday_appr_process_start estandarizada
new_dim_weekday_appr_process_start_data = [
    {'WEEKDAY_APPR_PROCESS_START_ID': 1, 'WEEKDAY_APPR_PROCESS_START': 'MONDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 2, 'WEEKDAY_APPR_PROCESS_START': 'TUESDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 3, 'WEEKDAY_APPR_PROCESS_START': 'WEDNESDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 4, 'WEEKDAY_APPR_PROCESS_START': 'THURSDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 5, 'WEEKDAY_APPR_PROCESS_START': 'FRIDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 6, 'WEEKDAY_APPR_PROCESS_START': 'SATURDAY'},
    {'WEEKDAY_APPR_PROCESS_START_ID': 7, 'WEEKDAY_APPR_PROCESS_START': 'SUNDAY'}
]

dim_weekday_appr_process_start_ref = pd.DataFrame(new_dim_weekday_appr_process_start_data)
display(dim_weekday_appr_process_start_ref)

,WEEKDAY_APPR_PROCESS_START_ID,WEEKDAY_APPR_PROCESS_START
0,1,MONDAY
1,2,TUESDAY
2,3,WEDNESDAY
3,4,THURSDAY
4,5,FRIDAY
5,6,SATURDAY
6,7,SUNDAY


In [18]:
# Categorias de la tabla dim_name_type_suite_ref.
csv_name = 'dim_name_type_suite.csv'
ref_table = 'previous_application'
dim_name_type_suite_ref = pd.read_csv(lookup_dir + ref_table + '/' + csv_name)
display(dim_name_type_suite_ref)

,NAME_TYPE_SUITE_ID,NAME_TYPE_SUITE
0,1,Unaccompanied
1,2,"Spouse, partner"
2,3,Family
3,4,Children
4,5,Other_B
5,6,Other_A
6,7,Group of people


### Incorporación de equivalencias sobre application_train.

In [38]:
# Lectura de application_train.
table_name = 'application_train'
csv_name = 'application_train.csv'
application_train = pd.read_csv(lookup_dir + table_name + '/' + csv_name)

In [39]:
# Conversion de valores de columnas ID a enteros.
for col in application_train.columns:
    if 'ID' in col:
        application_train[col] = application_train[col].astype('Int64')
        print(application_train[col].dtypes, end=', ')

Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, 

In [40]:
# Mapeo de name_contract_type.
name_contract_type_mapping = {
    1: 2, # Map original 'Cash loans' (ID 1) to new 'Cash loans' (ID 2)
    2: 3  # Map original 'Revolving loans' (ID 2) to new 'Revolving loans' (ID 3)
}

# Mapeo de weekday_start a dias de semana estandar.
weekday_mapping = {
    1: 3, # WEDNESDAY (original ID 1) -> WEDNESDAY (new ID 3)
    2: 1, # MONDAY (original ID 2)    -> MONDAY (new ID 1)
    3: 4, # THURSDAY (original ID 3)  -> THURSDAY (new ID 4)
    4: 7, # SUNDAY (original ID 4)    -> SUNDAY (new ID 7)
    5: 6, # SATURDAY (original ID 5)  -> SATURDAY (new ID 6)
    6: 5, # FRIDAY (original ID 6)    -> FRIDAY (new ID 5)
    7: 2  # TUESDAY (original ID 7)   -> TUESDAY (new ID 2)
}

# Mapeo de name_type_suite.
name_type_suite_mapping = {
    1: 1, # Map original 'Unaccompanied' (ID 1) to new 'Unaccompanied' (ID 1)
    2: 3,  # Map original 'Family' (ID 2) to new 'Family' (ID 3)
    3: 2,  # Map original 'Spouse, partner' (ID 3) to new 'Spouse, partner' (ID 2)
    4: 4,  # Map original 'Children' (ID 4) to new 'Children' (ID 4)
    5: 6,  # Map original 'Other_A' (ID 5) to new 'Other_A' (ID 6)
    6: 5,  # Map original 'Other_B' (ID 6) to new 'Other_B' (ID 5)
    7: 7   # Map original 'Group of people' (ID 7) to new 'Group of people' (ID 7)
}

# Armado de lista.
mappings = {'NAME_CONTRACT_TYPE_ID': name_contract_type_mapping,
            'WEEKDAY_APPR_PROCESS_START_ID': weekday_mapping,
            'NAME_TYPE_SUITE_ID': name_type_suite_mapping
}

In [41]:
# Loop para aplicacion de mapeo sobre application_train.
for dim_name, mapping in mappings.items():
    display(application_train[dim_name].value_counts()) # Mostrar conteo de valores antes de mapeo.
    application_train[dim_name] = application_train[dim_name].replace(mapping) # Aplicacion de mapeo sobre atributos de application_train.
    display(application_train[dim_name].value_counts()) # Mostrar conteo de valores despues de mapeo.

NAME_CONTRACT_TYPE_ID
1    278232
2     29279
Name: count, dtype: Int64

NAME_CONTRACT_TYPE_ID
2    278232
3     29279
Name: count, dtype: Int64

WEEKDAY_APPR_PROCESS_START_ID
7    53901
1    51934
2    50714
3    50591
6    50338
5    33852
4    16181
Name: count, dtype: Int64

WEEKDAY_APPR_PROCESS_START_ID
2    53901
3    51934
1    50714
4    50591
5    50338
6    33852
7    16181
Name: count, dtype: Int64

NAME_TYPE_SUITE_ID
1    248526
2     40149
3     11370
4      3267
6      1770
5       866
7       271
Name: count, dtype: Int64

NAME_TYPE_SUITE_ID
1    248526
3     40149
2     11370
4      3267
5      1770
6       866
7       271
Name: count, dtype: Int64

### Incorporación de equivalencias sobre previous_application.

In [42]:
# Lectura de previous_application.
table_name = 'previous_application'
csv_name = 'previous_application.csv'
previous_application = pd.read_csv(lookup_dir + table_name + '/' + csv_name)

In [43]:
# Conversion de valores de columnas ID a enteros.
for col in previous_application.columns:
    if 'ID' in col:
        previous_application[col] = previous_application[col].astype('Int64')
        print(previous_application[col].dtypes, end=', ')

Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, Int64, 

In [45]:
# Mapeo de name_contract_type.
name_contract_type_mapping = {} # No aplica, no existen modificaciones para realizar.

# Mapeo de weekday_start a dias de semana estandar.
weekday_mapping = {
    1: 6, # SATURDAY (original ID 1) -> SATURDAY (new ID 6)
    2: 4, # THURSDAY (original ID 2)    -> THURSDAY (new ID 4)
    3: 2, # TUESDAY (original ID 3)  -> TUESDAY (new ID 2)
    4: 1, # MONDAY (original ID 4)    -> MONDAY (new ID 1)
    5: 5, # FRIDAY (original ID 5)  -> FRIDAY (new ID 5)
    6: 7, # SUNDAY (original ID 6)    -> SUNDAY (new ID 7)
    7: 3  # WEDNESDAY (original ID 7)   -> WEDNESDAY (new ID 3)
}

# Mapeo de name_type_suite.
name_type_suite_mapping = {} # No aplica, no existen modificaciones para realizar.

# Armado de lista.
mappings = {'WEEKDAY_APPR_PROCESS_START_ID': weekday_mapping
}

In [46]:
# Loop para aplicacion de mapeo sobre application_train.
for dim_name, mapping in mappings.items():
    display(application_train[dim_name].value_counts()) # Mostrar conteo de valores antes de mapeo.
    application_train[dim_name] = application_train[dim_name].replace(mapping) # Aplicacion de mapeo sobre atributos de application_train.
    display(application_train[dim_name].value_counts()) # Mostrar conteo de valores despues de mapeo.

WEEKDAY_APPR_PROCESS_START_ID
2    53901
3    51934
1    50714
4    50591
5    50338
6    33852
7    16181
Name: count, dtype: Int64

WEEKDAY_APPR_PROCESS_START_ID
4    53901
2    51934
6    50714
1    50591
5    50338
7    33852
3    16181
Name: count, dtype: Int64

### Exportacion de tablas a csv y compresion a archivo zip

In [58]:
# Inicializacion de variables.
dir_name = 'overlap_fix'
output_dir = lookup_dir + dir_name + '/'
output_zip_file = lookup_dir + 'overlap_fix.zip'

In [59]:
# Funcion para zipeo.
def zip_folder(folder_path, output_path):
    """
    Compresses a folder into a zip file.
    """
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Add file to zip, preserving relative path within the zip
                zipf.write(file_path, os.path.relpath(file_path, os.path.dirname(folder_path)))
    print(f'Folder "{folder_path}" successfully zipped to "{output_path}".')

# Creacion de directorio output.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Tablas a exportar.
tables_to_export = {'application_train': application_train,
                    'previous_application': previous_application,
                    'dim_name_contract_type': dim_name_contract_type_ref,
                    'dim_weekday_appr_process_start': dim_weekday_appr_process_start_ref,
                    'dim_name_type_suite': dim_name_type_suite_ref
                    }

for table_name, table in tables_to_export.items():
    csv_name = f'{table_name}.csv'
    table.to_csv(output_dir + csv_name, index=False)
    print(f'DataFrame guardado como {csv_name}')

# Creacion de archivo zip.
zip_folder(output_dir, output_zip_file)

DataFrame guardado como application_train.csv
DataFrame guardado como previous_application.csv
DataFrame guardado como dim_name_contract_type.csv
DataFrame guardado como dim_weekday_appr_process_start.csv
DataFrame guardado como dim_name_type_suite.csv
Folder "../output/overlap_fix/" successfully zipped to "../output/overlap_fix.zip".
